# Modelagem Preditiva - Risco de Rebaixamento na Temporada Seguinte

## Objetivo

Este notebook trata da Fase 2 do projeto: treinar e avaliar um modelo de classificação
para a variável `rebaixado_prox_temporada`, construída e validada na Fase 1.

## Fonte dos dados

Base analítica clube-temporada, exportada do notebook 01 (EDA e preparação):
`data/processed/base_analitica_fase1.csv` (400 linhas, temporadas 2006-2025).


In [1]:
import pandas as pd

base = pd.read_csv("../data/processed/base_analitica_fase1.csv")
base.shape

(400, 13)

## Remoção de linhas sem variável-alvo definida

97 linhas não têm rebaixado_prox_temporada definido (clubes já rebaixados na própria
temporada, a temporada mais recente sem sucessora disponível, ou o caso administrativo
do Barueri em 2009). Essas linhas não podem ser usadas para treinar nem avaliar o
modelo e são removidas nesta etapa.


In [2]:
base_modelo = base.dropna(subset=["rebaixado_prox_temporada"]).copy()
base_modelo["rebaixado_prox_temporada"] = base_modelo["rebaixado_prox_temporada"].astype(int)
base_modelo.shape


(303, 13)

## Seleção de variáveis explicativas (features)

Das 13 colunas da base, algumas não podem ou não devem entrar como features:

- `rebaixado_prox_temporada`: é o alvo, não uma feature.
- `posicao_prox_temporada`: foi usada para _construir_ o alvo — incluí-la como feature
  seria vazamento de dados direto (a resposta estaria dentro da pergunta).
- `clube` e `temporada`: identificadores, não variáveis de desempenho. Usá-los faria o
  modelo memorizar "quais times" ao invés de aprender "quais padrões de desempenho"
  levam ao rebaixamento — o que não generaliza para temporadas futuras.
- `jogos`: é constante (38) em todas as linhas desde 2006, não carrega informação.

As oito colunas restantes — vitórias, empates, derrotas, gols pró, gols contra, pontos,
saldo de gols e posição — formam o conjunto de features desta primeira versão do
modelo.

O split treino/teste respeita a ordem temporal (Lones, 2021, citado no Referencial
Teórico da Fase 1): treino com temporadas até 2020, teste com 2021 em diante, sem
embaralhar.


In [4]:
features = ["vitorias", "empates", "derrotas", "gols_pro", "gols_contra",
            "pontos", "saldo_gols", "posicao"]

x = base_modelo[features]
y = base_modelo["rebaixado_prox_temporada"]

corte_temporal = base_modelo["temporada"] < 2021

x_treino, x_teste = x[corte_temporal], x[~corte_temporal]
y_treino, y_teste = y[corte_temporal], y[~corte_temporal]

x_treino.shape, x_teste.shape

((239, 8), (64, 8))

In [7]:
print("Treino: ", y_treino.shape[0], "- positivos: ", y_treino.sum(),
      f"({y_treino.mean()*100:.1f}%)")

print("Teste: ", y_teste.shape[0], "- positivos: ", y_teste.sum(),
      f"({y_teste.mean()*100:.1f}%)")

Treino:  239 - positivos:  39 (16.3%)
Teste:  64 - positivos:  11 (17.2%)


## Validação Temporal (Walk-Forward)

Um único corte treino/teste (2021) deixa apenas 64 observações e 11 rebaixamentos no teste, amostra pequena demais para uma métrica confiável (menos de 5 eventos por
variável no treino).

Para aproveitar melhor os dados sem violar a ordem temporal
(LONES, 2021, citado no Referencial Teórico), o modelo é avaliado por validação
cruzada com janela expansiva: treina com todas as temporadas anteriores a T e testa
na temporada T, repetindo para T de 2014 a 2024 (11 janelas). As previsões de todas
as janelas são agregadas antes de calcular as métricas finais.

Dado o desbalanceamento da variável-alvo (~17% de positivos), o modelo é treinado
com `class_weight="balanced"`, que prioriza identificar rebaixamentos reais (recall) em vez de apenas minimizar erros, escolha justificada pelo uso de negócio: um sistema de alerta de risco é mais útil quando captura a maioria das quedas reais, mesmo ao custo de mais falsos positivos.


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

temporadas = sorted(base_modelo["temporada"].unique())
inicio_teste = temporadas[8] #as primeiras temporadas para treino inicial (2006 - 20013)

y_real, y_pred = [], []

for t_teste in [t for t in temporadas if t >= inicio_teste]:
  treino = base_modelo[base_modelo["temporada"] < t_teste]
  teste = base_modelo[base_modelo["temporada"] == t_teste]

  modelo = LogisticRegression(max_iter=1000, class_weight="balanced")
  modelo.fit(treino[features], treino["rebaixado_prox_temporada"])
  pred = modelo.predict(teste[features])

  y_real.extend(teste["rebaixado_prox_temporada"].tolist())
  y_pred.extend(pred.tolist())

print("Precisão: ", round(precision_score(y_real, y_pred), 3))
print("Recall: ", round(recall_score(y_real, y_pred), 3))
print("F1: ", round(f1_score(y_real, y_pred), 3))
print(confusion_matrix(y_real, y_pred))

                       

Precisão:  0.26
Recall:  0.769
F1:  0.388
[[93 57]
 [ 6 20]]


### Resultado

O modelo de Regressão Logística, avaliado por validação cruzada temporal (11 janelas,
2014-2024, 176 previsões agregadas), obteve precisão de 0,26, recall de 0,77 e F1 de
0,39 com `class_weight="balanced"`. O modelo identifica corretamente 20 dos 26
rebaixamentos reais do período, ao custo de 57 falsos positivos. Essa configuração foi
escolhida em vez do modelo sem balanceamento (recall de apenas 0,15) por priorizar a
utilidade do KPI de risco como sistema de alerta: para o negócio, é preferível superestimar
risco em times seguros do que deixar de sinalizar times que de fato serão rebaixados.
